
# Distributed AI Damage Classification Service

## Notebook 01: Dataset Validation

### Goals
1. Access the MBDD2025 building surface defect dataset.
2. Inspect its files and annotation formats.
3. Identify available damage and structural-type labels.
4. Check class distributions and missing annotations.
5. Decide on a reliable training strategy before model development.

**Environment:** Google Colab  
**Stage:** Dataset validation — no model training yet.

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Creating a permanent project workspace
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DistributedAIProject")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print("Project workspace:", PROJECT_DIR)
print("Workspace exists:", PROJECT_DIR.exists())

Project workspace: /content/drive/MyDrive/DistributedAIProject
Workspace exists: True


In [ ]:
import os
import sys
import platform

In [ ]:
# Checking the runtime
print("Python version: ", sys.version.split()[0])
print("Operating system: ", platform.system())
print("Google Colab runtime: ", "COLAB_RELEASE_TAG" in os.environ)

Python version:  3.13.16
Operating system:  Linux
Google Colab runtime:  True


In [ ]:
# Checking GPU availability
import subprocess
import shutil

In [ ]:
if shutil.which("nvidia-smi") is not None:
  result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
  print(result.stdout)

else:
  print("nvidia-smi is unavailable.")
  print("Checking whether PyTorch can access a GPU...")

  try:
    import torch
    print("PyTorch version: ", torch.__version__)
    print("CUDA available: ", torch.cuda.is_available())

    if torch.cuda.is_available():
      print("GPU: ", torch.cuda.get_device_name(0))
    else:
      print("No GPU available. CPU is sufficient for dataset inspection.")

  except ImportError:
    print("PyTorch is not installed. No installation is needed yet.")

Wed Oct 28 15:57:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Installing lightweight inspection tools
%pip -q install requests beautifulsoup4

In [ ]:
# Checking the dataset record
import requests

In [ ]:
recordURL = "https://zenodo.org/api/records/15622584"

response = requests.get(recordURL, timeout=30)
response.raise_for_status()

record = response.json()

print("Dataset title: ", record["metadata"].get("title"))
print("Publication Date: ", record["metadata"].get("publication_date"))

print("\nAvailable files: ")
for i in record.get("files", []):
  print(
      "Name:", i.get("key"), "| Size (bytes):", i.get("size"), "| Download URL:", i.get("links", {}).get("self"))

Dataset title:  MBDD2025: A Deep Learning-Oriented Dataset for Diverse Building Defect Detection from UAV Images
Publication Date:  2025-06-09

Available files: 
Name: MBDD2025.zip | Size (bytes): 2533687033 | Download URL: https://zenodo.org/api/records/15622584/files/MBDD2025.zip/content


In [ ]:
# Checking the license and description
metadata = record.get("metadata", {})

print("Description: ")
print(metadata.get("description", "No description provided"))

print("\nLicense information:")
print(metadata.get("license", "No licence field found"))

print("\nKeywords:")
print(metadata.get("keywords", []))

Description: 
<p>MBDD2025 is a large-scale dataset of building surface defects collected by UAV imagery. It is designed for object detection and classification tasks in intelligent building inspection. The dataset covers a wide range of structural types and defect categories, making it suitable for deployment in diverse real-world scenarios.<br><br></p>

License information:
{'id': 'cc-by-4.0'}

Keywords:
['Building defect detection', 'Object detection', 'UAV imagery']


In [ ]:
# Checking remote access
import requests

In [ ]:
archiveURL = ("https://zenodo.org/api/records/15622584/"
              "files/MBDD2025.zip/content")
response = requests.head(archiveURL, allow_redirects=True, timeout=30)

In [ ]:
print("HTTP status: ", response.status_code)
print("Content length: ", response.headers.get("Content-Length"))
print("Accept-Ranges: ", response.headers.get("Accept-Ranges"))
print("Final URL: ", response.url)

response.close()

HTTP status:  200
Content length:  2533687033
Accept-Ranges:  None
Final URL:  https://zenodo.org/api/records/15622584/files/MBDD2025.zip/content


In [ ]:
# Checking Colab storage
import shutil

In [ ]:
total, used, free = shutil.disk_usage("/content")
def to_gb(n):
  return n/(1024**3)

print(f"Total disk space: {to_gb(total):.2f} GB")
print(f"Used disk space: {to_gb(total):.2f} GB")
print(f"Free disk space: {to_gb(free):.2f} GB")
print(f"Dataset archive: {2533687033/(1024**3):.2f} GB")

Total disk space: 112.64 GB
Used disk space: 112.64 GB
Free disk space: 70.06 GB
Dataset archive: 2.36 GB


In [ ]:
for i in ["/content", "/tmp"]:
    usage = shutil.disk_usage(i)

    print(f"\nPath: {i}")
    print(f"Total: {usage.total/1024**3:.2f} GiB")
    print(f"Used:  {usage.used/1024**3:.2f} GiB")
    print(f"Free:  {usage.free/1024**3:.2f} GiB")

print("\nDataset archive: approximately 2.36 GiB")


Path: /content
Total: 112.64 GiB
Used:  42.56 GiB
Free:  70.06 GiB

Path: /tmp
Total: 112.64 GiB
Used:  42.56 GiB
Free:  70.06 GiB

Dataset archive: approximately 2.36 GiB


In [12]:
# Downloading MBDD2025
import requests
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DistributedAIProject")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

archive_path = PROJECT_DIR / "MBDD2025.zip"

# Official Zenodo file-download endpoint
download_url = ("https://zenodo.org/records/15622584/files/MBDD2025.zip?download=1")

if archive_path.exists() and archive_path.stat().st_size > 0:
    print("Archive already exists:", archive_path)
    print("Size (GB):", round(archive_path.stat().st_size / 1e9, 2))

else:
    with requests.get(download_url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        with open(archive_path, "wb") as out:
            for i in response.iter_content(chunk_size=8 * 1024 * 1024):
                if i:
                    out.write(i)
    print("Download complete:", archive_path)
    print("Size (GB):", round(archive_path.stat().st_size / 1e9, 2))


Archive already exists: /content/drive/MyDrive/DistributedAIProject/MBDD2025.zip
Size (GB): 2.53


In [13]:
# Verifying the ZIP
import zipfile
from pathlib import Path

archive_path = Path("/content/drive/MyDrive/DistributedAIProject/MBDD2025.zip")

print("Archive exists:", archive_path.exists())
print("Archive size (GB):", round(archive_path.stat().st_size / 1e9, 2))

with zipfile.ZipFile(archive_path, "r") as f:
    print("ZIP integrity check: ", f.testzip())
    print("Number of XML annotations:", sum(i.startswith("MBDD2025/Annotations/") and i.endswith(".xml") for i in f.namelist()))

Archive exists: True
Archive size (GB): 2.53
ZIP integrity check:  None
Number of XML annotations: 14471


In [ ]:
# Performing an inspection on the ZIP file
import zipfile

In [ ]:
archive_path = "/content/MBDD2025.zip"
with zipfile.ZipFile(archive_path, "r") as f:
  files = f.infolist()
  n = len(files)

  print("Total entries: ", n)
  print("\nFirst 40 archive entries.")

  for i in files[:40]:
    print(i.filename, "|", i.file_size, "bytes")

  print("\nPotential annotation and documentation files:")
  for i in files:
    name = i.filename.lower()
    if (name.endswith((".xml", ".yaml", ".yml", ".txt", ".json")) and any (j in name for j in ("annotation", "label", "class", "readme", "data.yaml", "dataset"))):
      print(i.filename)

Total entries:  43418

First 40 archive entries.
MBDD2025/ | 0 bytes
MBDD2025/Annotations/ | 0 bytes
MBDD2025/Annotations/Hefei1.xml | 775 bytes
MBDD2025/Annotations/Hefei10.xml | 776 bytes
MBDD2025/Annotations/Hefei100.xml | 2423 bytes
MBDD2025/Annotations/Hefei1000.xml | 1187 bytes
MBDD2025/Annotations/Hefei10000.xml | 1575 bytes
MBDD2025/Annotations/Hefei10001.xml | 1983 bytes
MBDD2025/Annotations/Hefei10002.xml | 3612 bytes
MBDD2025/Annotations/Hefei10003.xml | 4043 bytes
MBDD2025/Annotations/Hefei10004.xml | 1578 bytes
MBDD2025/Annotations/Hefei10005.xml | 2812 bytes
MBDD2025/Annotations/Hefei10006.xml | 3214 bytes
MBDD2025/Annotations/Hefei10007.xml | 3228 bytes
MBDD2025/Annotations/Hefei10008.xml | 4860 bytes
MBDD2025/Annotations/Hefei10009.xml | 3628 bytes
MBDD2025/Annotations/Hefei1001.xml | 1188 bytes
MBDD2025/Annotations/Hefei10010.xml | 4848 bytes
MBDD2025/Annotations/Hefei10011.xml | 5657 bytes
MBDD2025/Annotations/Hefei10012.xml | 5254 bytes
MBDD2025/Annotations/Hefei1001

In [ ]:
# Inspecting one XML annotation
from pathlib import Path
archive_path = "/content/MBDD2025.zip"

with zipfile.ZipFile(archive_path, "r") as f:
  xml_path = "MBDD2025/Annotations/Hefei100.xml"
  xml_content = f.read(xml_path).decode("utf-8", errors="replace")

print(xml_content)

<?xml version='1.0' encoding='utf-8'?>
<annotation>
  <folder>/dji</folder>
  <filename>100.jpg</filename>
  <source>
    <database>Unknown</database>
    <annotation>Unknown</annotation>
    <image>Unknown</image>
  </source>
  <size>
    <width>1280</width>
    <height>720</height>
    <depth />
  </size>
  <segmented>0</segmented>
  <object>
    <name>corrosion</name>
    <truncated>0</truncated>
    <occluded>0</occluded>
    <difficult>0</difficult>
    <bndbox>
      <xmin>2.38</xmin>
      <ymin>174.69</ymin>
      <xmax>44.21</xmax>
      <ymax>290.87</ymax>
    </bndbox>
    <attributes>
      <attribute>
        <name>rotation</name>
        <value>0.0</value>
      </attribute>
    </attributes>
  </object>
  <object>
    <name>corrosion</name>
    <truncated>0</truncated>
    <occluded>0</occluded>
    <difficult>0</difficult>
    <bndbox>
      <xmin>93.01</xmin>
      <ymin>243.24</ymin>
      <xmax>242.89</xmax>
      <ymax>293.2</ymax>
    </bndbox>
    <attributes>
   

In [ ]:
# Discovering all damage classes
import xml.etree.ElementTree as ET
from collections import Counter

archive_path = "/content/MBDD2025.zip"

damageCounts = Counter()
annotationCount = 0
objects_without_labels = 0
xml_errors = 0

with zipfile.ZipFile(archive_path, "r") as f:
  xmlFiles = [i for i in f.namelist() if i.lower().endswith(".xml") and "/annotations/" in i.lower()]
  for j in xmlFiles:
    try:
      root = ET.fromstring(f.read(j))
      annotationCount = annotationCount+1
      for k in root.findall("object"):
        label = k.findtext("name", default="").strip()
        if label:
          damageCounts[label] = damageCounts[label]+1
        else:
          objects_without_labels = objects_without_labels+1

    except ET.ParseError:
      xml_errors = xml_errors+1

print("XML annotation files processed: ", annotationCount)
print("XML parsing errors: ", xml_errors)
print("Objects without labels: ", objects_without_labels)

print("\nDamage classes and object counts: ")
for label, count in damageCounts.most_common():
  print(f"{label}: {count:,}")

print("\nTotal annotated objects: ", sum(damageCounts.values()))

XML annotation files processed:  14471
XML parsing errors:  0
Objects without labels:  0

Damage classes and object counts: 
abscission: 22,702
crack: 17,044
corrosion: 9,207
leakage: 6,642
bulge: 2,018

Total annotated objects:  57613


In [ ]:
with zipfile.ZipFile(archive_path, "r") as f:
  names = f.namelist()

  print("TOP-LEVEL CONTENTS")
  top_level = sorted({i.split("/")[1] for i in names if i.startswith("MBDD2025/") and len(i.split("/"))>2})
  for i in top_level:
    print("-", i)

  print("\nREADME / METADATA FILES")
  docs = [i for i in names if i.lower().endswith(("readme.md", "readme.txt", "metadata.json", "data.yaml")) and not i.endswith("/")]
  for i in docs:
    print(f"\n--- {i} ---")
    try:
      content = f.read(i).decode("utf-8", errors="replace")
      print(content[:8000])
    except Exception as e:
      print("Could not read: ", e)

TOP-LEVEL CONTENTS
- Annotations
- JPEGImages
- Labels

README / METADATA FILES

--- MBDD2025/README.md ---
# MBDD2025: A Deep Learning-Oriented Dataset for Diverse Building Defect Detection from UAV Images

## 📂 Dataset Overview

**MBDD2025** is a large-scale dataset of building wall defects collected via UAV imagery. The dataset is designed for object detection and classification tasks in the field of intelligent building inspection. It includes a wide range of structural types and defect categories, making it suitable for generalization across various real-world scenarios.

- **Total images**: 14,471
- **Image resolution**: High-resolution UAV photos
- **Defect types (5)**:  
  - Crack  
  - Leakage  
  - Abscission  
  - Corrosion  
  - Bulge 

- **Building structure types (6)**:  
  - Steel structure  
  - Wooden structure
  - Masonry structure
  - Brick and wood structure
  - Brick concrete structure  
  - Reinforced concrete structure  

- **Annotation format**: VOC XML  
- **Co

In [ ]:
import zipfile
archive_path = "/content/MBDD2025.zip"

with zipfile.ZipFile(archive_path, "r") as f:
  label_files = [i for i in f.infolist() if i.filename.startswith("MBDD2025/Labels/") and not i.is_dir()]
  print("Number of files in labels = ", len(label_files))

  for i in label_files[:40]:
    print(f"{i.filename} | {i.file_size:,} bytes")

  for i in label_files:
    if i.file_size<=100_000:
      if i.filename.lower().endswith((".txt", ".csv", ".json", ".yaml", ".yml")):
        print(f"\n--- {i.filename} ---")
        print(f.read(i.filename).decode("utf-8", errors="replace")[:5000])

  if not label_files:
    print("\nNo files found inside the Labels/ directory.")

Streaming output truncated to the last 5000 lines.

--- MBDD2025/Labels/Hefei9422.txt ---
2 0.5 0.7082291666666666 1.0 0.17668055555555554
2 0.5810195312499999 0.10992361111111112 0.17646093750000003 0.21379166666666669


--- MBDD2025/Labels/Hefei9423.txt ---
2 0.54507421875 0.09947222222222223 0.14116406249999996 0.1742777777777778
2 0.49925390625 0.7162638888888889 0.9985078125 0.2248611111111111


--- MBDD2025/Labels/Hefei9424.txt ---
2 0.536578125 0.09330555555555557 0.14770312500000005 0.18661111111111114
2 0.487984375 0.7376180555555556 0.97596875 0.2081527777777778


--- MBDD2025/Labels/Hefei9425.txt ---
2 0.5 0.7202777777777777 1.0 0.2891111111111111
2 0.5248125 0.09714583333333333 0.15815625000000005 0.19287500000000002


--- MBDD2025/Labels/Hefei9426.txt ---
2 0.5 0.6855833333333333 1.0 0.2544166666666668
2 0.5104335937500001 0.09446527777777779 0.16600781250000002 0.18893055555555557


--- MBDD2025/Labels/Hefei9427.txt ---
2 0.5 0.6855833333333333 1.0 0.2544166666666668
2 0.

In [ ]:
from pathlib import Path

archive_path = Path("/content/MBDD2025.zip")

if archive_path.exists():
    size_gb = archive_path.stat().st_size / (1024**3)
    print(f"Partial or complete download: {size_gb:.2f} GiB")
    print(f"Size in bytes: {archive_path.stat().st_size:,}")
else:
    print("No downloaded archive found.")

Partial or complete download: 1.83 GiB
Size in bytes: 1,970,274,304


In [ ]:
import requests

url = (
    "https://zenodo.org/api/records/15622584/"
    "files/MBDD2025.zip/content"
)

response = requests.get(url, headers={"Range": "bytes=1970274304-1970274304"}, stream=True, timeout=30)

print("HTTP status:", response.status_code)
print("Content-Range:", response.headers.get("Content-Range"))
print("Accept-Ranges:", response.headers.get("Accept-Ranges"))

response.close()

HTTP status: 206
Content-Range: bytes 1970274304-1970274304/2533687033
Accept-Ranges: bytes


In [ ]:
import requests
from pathlib import Path

url = (
    "https://zenodo.org/api/records/15622584/"
    "files/MBDD2025.zip/content"
)

path = Path("/content/MBDD2025.zip")
expected_size = 2533687033

start = path.stat().st_size

if start == expected_size:
    print("Archive is already complete!")

elif start > expected_size:
    raise RuntimeError("File is larger than expected. Stopping.")

else:
    headers = {"Range": f"bytes={start}-"}
    with requests.get(url, headers=headers, stream=True, timeout=(30, 120)) as response:
        content_range = response.headers.get("Content-Range", "")
        expected_prefix = f"bytes {start}-"

        if response.status_code != 206:
            raise RuntimeError(f"Expected HTTP 206, got {response.status_code}. Existing file was not modified.")

        if not content_range.startswith(expected_prefix):
            raise RuntimeError(f"Unexpected Content-Range: {content_range}. Existing file was not modified.")

        print("Resuming from:", f"{start:,}", "bytes")
        with path.open("ab") as f:
            for i in response.iter_content(chunk_size=1024*1024):
                if i:
                    f.write(i)
                    start = start+len(i)
                    if start % (100*1024*1024) < len(i):
                        print(f"Downloaded: {start / 1024**3:.2f} GiB")

    actual_size = path.stat().st_size
    print(f"\nFinal size: {actual_size:,} bytes")

    if actual_size == expected_size:
        print("SUCCESS: Archive size verified!")
    else:
        print("Download incomplete. Rerun this cell to resume.")

Resuming from: 1,970,274,304 bytes
Downloaded: 1.86 GiB
Downloaded: 1.95 GiB
Downloaded: 2.05 GiB
Downloaded: 2.15 GiB
Downloaded: 2.25 GiB
Downloaded: 2.34 GiB

Final size: 2,533,687,033 bytes
SUCCESS: Archive size verified!


In [ ]:
# Verifying the ZIP integrity
import zipfile

path = "/content/MBDD2025.zip"

with zipfile.ZipFile(path, "r") as f:
  badFile = f.testzip()

if badFile is None:
  print("ZIP integrity verified. Ready to continue!")
else:
  print("Corrupted archive member: ", badFile)

ZIP integrity verified. Ready to continue!


In [ ]:
# Finding the class mapping
with zipfile.ZipFile(archive_path, "r") as f:
  for i in f.namelist():
    lower = i.lower()
    if (i.startswith("MBDD2025/") and not i.endswith("/") and any(j in lower for j in ("classes.txt", "class_names", "data.yaml", "dataset.yaml", "labels.txt", "metadata"))):
      print("\nFILE: ", i)
      info = f.getinfo(i)

      if info.file_size <= 20000:
        print(f.read(i).decode("utf-8", errors="replace"))
      else:
        print("File too large to preview.")

In [ ]:
import zipfile

archive_path = "/content/MBDD2025.zip"
with zipfile.ZipFile(archive_path, "r") as f:
    names = f.namelist()
    matches = [i for i in names if any(j in j.lower() for j in [
            "classes.txt",
            "class_names",
            "data.yaml",
            "dataset.yaml",
            "labels.txt",
            "metadata"
        ]) and not j.endswith("/")
    ]

    if matches:
        print("Possible class-mapping files found:")
        for i in matches:
            print("\nFILE:", i)
            if f.getinfo(i).file_size <= 20000:
                print(f.read(i).decode("utf-8", errors="replace"))
    else:
        print("No obvious class-mapping files found.")
        print("Next, we will inspect the numeric class IDs.")

Streaming output truncated to the last 5000 lines.
2 0.6306875000000001 0.10760416666666667 0.30456250000000007 0.21379166666666669


FILE: MBDD2025/Labels/Hefei9439.txt
2 0.5 0.7087152777777778 1.0 0.1619027777777777
2 0.63134375 0.11805555555555557 0.32939062500000005 0.17427777777777778
2 0.36338281250000004 0.09563194444444445 0.15946875000000005 0.1912638888888889


FILE: MBDD2025/Labels/Hefei944.txt
1 0.65919921875 0.49659722222222225 0.08319531250000001 0.9931944444444445
1 0.56521875 0.7013333333333334 0.07723437500000002 0.49930555555555567
1 0.34557421875 0.7286180555555556 0.0854609375 0.28031944444444445
1 0.2920078125 0.3338125 0.04573437500000002 0.29956944444444444
1 0.22701171875 0.25143055555555555 0.055367187500000005 0.5028611111111111
1 0.23846093750000003 0.7692708333333333 0.05895312499999999 0.3873194444444444


FILE: MBDD2025/Labels/Hefei9440.txt
2 0.33855078125000004 0.10295833333333335 0.18299218750000001 0.19519444444444445
2 0.6012773437500001 0.101798611111

In [ ]:
import zipfile

archive_path = "/content/MBDD2025.zip"

with zipfile.ZipFile(archive_path, "r") as f:
    # Find the first few label files
    label_files = [i for i in f.namelist() if i.startswith("MBDD2025/Labels/") and i.endswith(".txt")][:5]
    print("Sample annotation files:\n")
    for j in label_files:
        print(f"--- {j} ---")
        content = f.read(j).decode("utf-8", errors="replace")
        print(repr(content[:500]))
        print()

Sample annotation files:

--- MBDD2025/Labels/Hefei1.txt ---
'3 0.5084765625000001 0.35477083333333337 0.06426562499999999 0.15104166666666669\r\n'

--- MBDD2025/Labels/Hefei10.txt ---
'3 0.52306640625 0.3470277777777778 0.05882031249999997 0.13166666666666665\r\n'

--- MBDD2025/Labels/Hefei100.txt ---
'3 0.018199218750000003 0.32330555555555557 0.0326796875 0.16136111111111112\r\n3 0.13121093749999999 0.37252777777777785 0.11709375 0.06938888888888886\r\n3 0.24467578124999997 0.3152361111111111 0.035398437500000005 0.05486111111111111\r\n3 0.51834765625 0.3225 0.043570312499999986 0.06294444444444448\r\n3 0.58143359375 0.3692986111111111 0.03358593750000001 0.06293055555555556\r\n'

--- MBDD2025/Labels/Hefei1000.txt ---
'1 0.68452734375 0.5966111111111111 0.08425781250000003 0.8067777777777778\r\n2 0.30432421875000004 0.40232638888888883 0.0633828125 0.12631944444444443\r\n'

--- MBDD2025/Labels/Hefei10000.txt ---
'0 0.10098828124999999 0.6375069444444444 0.20066406250000002 0.1559583

In [ ]:
# Finding the class names
with zipfile.ZipFile(archive_path, "r") as f:
    # Looking for the dataset README and annotation metadata
    candidates = [i for i in f.namelist() if i.lower().endswith(("readme.md", "data.yaml", "dataset.yaml", "classes.txt"))]

    print("Metadata files found:", candidates)
    for i in candidates:
        print(f"\n--- {i} ---")
        content = f.read(i).decode("utf-8", errors="replace")
        print(content[:5000])

    # Inspecting an XML annotation to find the original defect class names
    xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")]
    if xml_files:
        print("\n--- Sample XML annotation ---")
        xml = f.read(xml_files[0]).decode("utf-8", errors="replace")
        print(xml[:3000])

Metadata files found: ['MBDD2025/README.md']

--- MBDD2025/README.md ---
# MBDD2025: A Deep Learning-Oriented Dataset for Diverse Building Defect Detection from UAV Images

## 📂 Dataset Overview

**MBDD2025** is a large-scale dataset of building wall defects collected via UAV imagery. The dataset is designed for object detection and classification tasks in the field of intelligent building inspection. It includes a wide range of structural types and defect categories, making it suitable for generalization across various real-world scenarios.

- **Total images**: 14,471
- **Image resolution**: High-resolution UAV photos
- **Defect types (5)**:  
  - Crack  
  - Leakage  
  - Abscission  
  - Corrosion  
  - Bulge 

- **Building structure types (6)**:  
  - Steel structure  
  - Wooden structure
  - Masonry structure
  - Brick and wood structure
  - Brick concrete structure  
  - Reinforced concrete structure  

- **Annotation format**: VOC XML  
- **Compression format**: `.zip`

---

##

In [ ]:
# Verifying every defect class and check for structural labels
defect_counts = Counter()
structure_terms = Counter()
xml_count = 0
xml_errors = 0

possible_structure_labels = {
    "steel structure",
    "wooden structure",
    "masonry structure",
    "brick and wood structure",
    "brick concrete structure",
    "reinforced concrete structure",
}

with zipfile.ZipFile(archive_path, "r") as f:
  xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.lower().endswith(".xml")]
  xml_count = len(xml_files)

  for j in xml_files:
    try:
      root = ET.fromstring(f.read(j))

      # Counting object-level defect labels
      for k in root.findall(".//object"):
        name = k.findtext("name", default="").strip().lower()
        if name:
          defect_counts[name] = defect_counts[name]+1

      # Checking all XML text for explicit structure labels
      xml_text = " ".join(text.strip().lower() for text in root.itertext() if text and text.strip())
      for k in possible_structure_labels:
        if k in xml_text:
          structure_terms[k] = structure_terms[k]+1

    except ET.ParseError:
      xml_errors = xml_errors+1

print("XML annotation files:", xml_count)
print("XML parsing errors:", xml_errors)

print("\nDefect class counts (annotated objects):")
for name, count in sorted(defect_counts.items()):
    print(f"{name}: {count}")

print("\nXML files containing explicit structure-type terms:")
if structure_terms:
    for name, count in sorted(structure_terms.items()):
        print(f"{name}: {count} files")
else:
    print("No explicit structure-type terms found.")

XML annotation files: 14471
XML parsing errors: 0

Defect class counts (annotated objects):
abscission: 22702
bulge: 2018
corrosion: 9207
crack: 17044
leakage: 6642

XML files containing explicit structure-type terms:
No explicit structure-type terms found.


In [ ]:
from pathlib import PurePosixPath

In [ ]:
with zipfile.ZipFile(archive_path, "r") as f:
  names = f.namelist()
  images = [i for i in names if i.startswith("MBDD2025/JPEGImages/") and PurePosixPath(i).suffix.lower() in {".jpg", ".jpeg", ".png"}]
  annotations = [i for i in names if i.startswith("MBDD2025/Annotations/") and i.lower().endswith(".xml")]

  imageStems = Counter(PurePosixPath(i).stem for i in images)
  annotationStems = Counter(PurePosixPath(i).stem for i in annotations)

  print("Images: ", len(images))
  print("XML annotations: ", len(annotations))
  print("Images without matching XML: ", len(imageStems-annotationStems))
  print("XML without matching image: ", len(annotationStems-imageStems))
  print("\nSample image paths: ")
  print(*images[:5], sep="\n")
  print("\nSample annotation paths: ")
  print(*annotations[:5], sep="\n")

Images:  14471
XML annotations:  14471
Images without matching XML:  0
XML without matching image:  0

Sample image paths: 
MBDD2025/JPEGImages/Hefei1.jpg
MBDD2025/JPEGImages/Hefei10.jpg
MBDD2025/JPEGImages/Hefei100.jpg
MBDD2025/JPEGImages/Hefei1000.jpg
MBDD2025/JPEGImages/Hefei10000.jpg

Sample annotation paths: 
MBDD2025/Annotations/Hefei1.xml
MBDD2025/Annotations/Hefei10.xml
MBDD2025/Annotations/Hefei100.xml
MBDD2025/Annotations/Hefei1000.xml
MBDD2025/Annotations/Hefei10000.xml


In [28]:
import zipfile
import xml.etree.ElementTree as ET

from pathlib import PurePosixPath
from PIL import Image
from io import BytesIO
from collections import Counter

archive_path = "/content/MBDD2025.zip"

checked = 0
badImages = []
badBoxes = []
emptyAnnotations = 0
defect_counts = Counter()
image_sizes = Counter()

with zipfile.ZipFile(archive_path, "r") as f:
  xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.lower().endswith(".xml")]
  for j in xml_files:
    stem = PurePosixPath(j).stem
    image_name = f"MBDD2025/JPEGImages/{stem}.jpg"
    try:
      image = Image.open(BytesIO(f.read(image_name)))
      image.verify()
      image = Image.open(BytesIO(f.read(image_name)))
      width, height = image.size
      image_sizes[(width,height)] = image_sizes[(width,height)]+1
      root = ET.fromstring(f.read(j))
      objects = root.findall("object")
      if objects:
        emptyAnnotations = emptyAnnotations+1
        for k in objects:
          label = k.findtext("name", "").strip().lower()
          defect_counts[label] = defect_counts[label]+1
          box = k.find("bndbox")
          if box is None:
            badBoxes.append((j, "missing box"))
            continue

          xmin = float(box.findtext("xmin"))
          ymin = float(box.findtext("ymin"))

          xmax = float(box.findtext("xmax"))
          ymax = float(box.findtext("ymax"))

          if not (0<=xmin<xmax<=width and 0<=ymin<ymax<=height):
            badBoxes.append((j, (xmin, ymin, xmax, ymax), (width, height)))
        checked = checked+1

    except Exception as e:
      badImages.append((image_name, str(e)))

print("Images and annotations checked:", checked)
print("Image decoding/errors:", len(badImages))
print("Invalid bounding boxes:", len(badBoxes))
print("Images with no annotated objects:", emptyAnnotations)

print("\nImage dimensions (top 10):")
for size, count in image_sizes.most_common(10):
    print(f"{size}: {count}")

print("\nDefect instance counts:")
for label, count in sorted(defect_counts.items()):
    print(f"{label}: {count}")

print("\nSample errors:")
print("Images:", badImages[:3])
print("Boxes:", badBoxes[:3])

Images and annotations checked: 14463
Image decoding/errors: 0
Invalid bounding boxes: 116
Images with no annotated objects: 14463

Image dimensions (top 10):
(1280, 720): 14102
(720, 1280): 254
(960, 544): 115

Defect instance counts:
abscission: 22702
bulge: 2018
corrosion: 9207
crack: 17044
leakage: 6642

Sample errors:
Images: []
Boxes: [('MBDD2025/Annotations/Hefei11420.xml', (-7.75, 135.09, 201.8, 165.1), (1280, 720)), ('MBDD2025/Annotations/Hefei11421.xml', (-18.96, 127.34, 226.97, 158.32), (1280, 720)), ('MBDD2025/Annotations/Hefei11422.xml', (-9.97, 118.36, 216.08, 153.73), (1280, 720))]


In [29]:
import zipfile
import xml.etree.ElementTree as ET

from collections import Counter

archive_path = "/content/MBDD2025.zip"

with zipfile.ZipFile(archive_path, "r") as f:
    xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")]

    empty_count = 0
    nonempty_count = 0
    object_counts = Counter()

    for j in xml_files:
        root = ET.fromstring(f.read(j))
        objects = root.findall("object")

        if len(objects) == 0:
            empty_count = empty_count+1

        else:
            nonempty_count = nonempty_count+1
            object_counts[len(objects)] = object_counts[len(objects)]+1

    print("Total XML files:", len(xml_files))
    print("Empty annotations:", empty_count)
    print("Annotations with objects:", nonempty_count)
    print("Object counts per image:", object_counts.most_common(10))

    sample = ET.fromstring(f.read("MBDD2025/Annotations/Hefei11420.xml"))

    print("\nHefei11420 object names:")
    for k in sample.findall("object"):
        print(k.findtext("name"), k.find("bndbox") is not None)

Total XML files: 14471
Empty annotations: 8
Annotations with objects: 14463
Object counts per image: [(1, 4081), (2, 3560), (3, 1848), (4, 1257), (5, 779), (6, 729), (7, 401), (10, 230), (8, 225), (9, 225)]

Hefei11420 object names:
abscission True
abscission True
crack True
abscission True
abscission True
abscission True
abscission True
abscission True
abscission True
abscission True
abscission True
crack True


In [30]:
# Inspecting the 8 empty annotations
import zipfile
import xml.etree.ElementTree as ET

In [31]:
from pathlib import PurePosixPath

In [33]:
with zipfile.ZipFile(archive_path, "r") as f:
  for i in f.namelist():
    if (i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")):
      root = ET.fromstring(f.read(i))
      if not root.findall("object"):
        stem = PurePosixPath(i).stem
        image_name = f"MBDD2025/JPEGImages/{stem}.jpg"

        print("Empty annotation:", i)
        print("Matching image exists:", image_name in f.namelist())
        print()

Empty annotation: MBDD2025/Annotations/Hefei10237.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei10278.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei10421.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei13256.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei1821.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei425.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei901.xml
Matching image exists: True

Empty annotation: MBDD2025/Annotations/Hefei9028.xml
Matching image exists: True



In [35]:
# Inspecting the 116 invalid bounding boxes
import zipfile
import xml.etree.ElementTree as ET

from collections import Counter

archive_path = "/content/MBDD2025.zip"

invalid_count = 0
clippable_count = 0
unusable_count = 0
issues_by_type = Counter()
examples = []

with zipfile.ZipFile(archive_path, "r") as f:
    xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")]
    for j in xml_files:
        root = ET.fromstring(f.read(j))
        size = root.find("size")
        if size is None:
            continue

        width = int(size.findtext("width"))
        height = int(size.findtext("height"))

        for k in root.findall("object"):
            box = k.find("bndbox")
            if box is None:
                continue

            xmin = float(box.findtext("xmin"))
            ymin = float(box.findtext("ymin"))

            xmax = float(box.findtext("xmax"))
            ymax = float(box.findtext("ymax"))

            if not (0<=xmin<xmax<=width and 0<=ymin<ymax<=height):
                invalid_count = invalid_count+1

                clipped_xmin = max(0, min(xmin, width))
                clipped_ymin = max(0, min(ymin, height))
                clipped_xmax = max(0, min(xmax, width))
                clipped_ymax = max(0, min(ymax, height))

                if (clipped_xmin<clipped_xmax and clipped_ymin<clipped_ymax):
                    clippable_count = clippable_count+1
                    issue = "clippable"

                else:
                    unusable_count = unusable_count-1
                    issue = "unusable"

                issues_by_type[issue] = issues_by_type[issue]+1

                if len(examples) < 10:
                    examples.append((j, k.findtext("name"), (xmin, ymin, xmax, ymax), (clipped_xmin, clipped_ymin, clipped_xmax, clipped_ymax)))

print("Invalid boxes:", invalid_count)
print("Boxes that remain valid after clipping:", clippable_count)
print("Boxes unusable after clipping:", unusable_count)

print("\nFirst 10 examples:")
for i in examples:
    print(i)

Invalid boxes: 116
Boxes that remain valid after clipping: 116
Boxes unusable after clipping: 0

First 10 examples:
('MBDD2025/Annotations/Hefei11420.xml', 'crack', (-7.75, 135.09, 201.8, 165.1), (0, 135.09, 201.8, 165.1))
('MBDD2025/Annotations/Hefei11421.xml', 'crack', (-18.96, 127.34, 226.97, 158.32), (0, 127.34, 226.97, 158.32))
('MBDD2025/Annotations/Hefei11422.xml', 'crack', (-9.97, 118.36, 216.08, 153.73), (0, 118.36, 216.08, 153.73))
('MBDD2025/Annotations/Hefei11423.xml', 'crack', (-14.12, 110.88, 270.54, 152.51), (0, 110.88, 270.54, 152.51))
('MBDD2025/Annotations/Hefei11424.xml', 'crack', (-13.15, 103.13, 247.29999999999998, 143.8), (0, 103.13, 247.29999999999998, 143.8))
('MBDD2025/Annotations/Hefei11425.xml', 'crack', (-5.79, 111.0, 259.89, 147.8), (0, 111.0, 259.89, 147.8))
('MBDD2025/Annotations/Hefei11426.xml', 'crack', (-18.4, 93.45, 280.22, 150.58), (0, 93.45, 280.22, 150.58))
('MBDD2025/Annotations/Hefei11427.xml', 'crack', (-16.96, 95.0, 279.99, 146.31), (0, 95.0, 2

In [37]:
# Performing the 80/10/10 split
# 80% Training, 10% Validation, 10% Testing
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

In [38]:
from pathlib import PurePosixPath
from collections import Counter
from sklearn.model_selection import train_test_split

In [40]:
archive_path = "/content/MBDD2025.zip"
random_seed = 42

class_names = ["abscission", "bulge", "corrosion", "crack", "leakage"]
records = []

excludedEmpty = 0
excludedMulti = 0
excludedUnknown = 0

with zipfile.ZipFile(archive_path, "r") as f:
  xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")]
  for j in xml_files:
    root = ET.fromstring(f.read(j))
    objects = root.findall("object")

    labels = {
        k.findtext("name", "").strip().lower() for k in objects if k.findtext("name", "").strip()
    }

    if not labels:
      excludedEmpty = excludedEmpty+1
      continue

    if not labels.issubset(set(class_names)):
      excludedUnknown = excludedUnknown+1
      continue

    if len(labels)!=1:
      excludedMulti = excludedMulti+1
      continue

    image_name = PurePosixPath(j).stem+".jpg"
    records.append(
        {
            "image": image_name,
            "label": next(iter(labels))
        }
    )

  load = pd.DataFrame(records)

# Split 80% train, 20% temporary
train, temp = train_test_split(load, test_size=0.20, random_state=random_seed, stratify=load["label"],)

# Splitting temporary data equally into validation and test
validation, test = train_test_split(temp, test_size=0.50, random_state=random_seed, stratify=temp["label"],)

train = train.copy()
validation = validation.copy()
test = test.copy()

train["split"] = "train"
validation["split"] = "validation"
test["split"] = "test"

manifest = pd.concat([train, validation, test], ignore_index=True,)
manifest.to_csv("/content/mbdd_classification_manifest.csv", index=False,)

print("Excluded empty annotations:", excludedEmpty)
print("Excluded multi-category images:", excludedMulti)
print("Excluded unknown-label images:", excludedUnknown)

print("\nSplit sizes:")
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))
print("Total:", len(manifest))

print("\nClass distribution by split:")
print(pd.crosstab(manifest["label"], manifest["split"]))

print("\nManifest saved to:")
print("/content/mbdd_classification_manifest.csv")

Excluded empty annotations: 8
Excluded multi-category images: 2399
Excluded unknown-label images: 0

Split sizes:
Train: 9651
Validation: 1206
Test: 1207
Total: 12064

Class distribution by split:
split       test  train  validation
label                              
abscission   334   2672         334
bulge         75    600          75
corrosion    166   1330         166
crack        437   3490         436
leakage      195   1559         195

Manifest saved to:
/content/mbdd_classification_manifest.csv


In [41]:
import pandas as pd

manifest = pd.read_csv("/content/mbdd_classification_manifest.csv")

print("Total rows:", len(manifest))
print("Unique image filenames:", manifest["image"].nunique())
print("Duplicate filenames:", manifest["image"].duplicated().sum())

print("\nImages per split:")
print(manifest["split"].value_counts())

print("\nImages assigned to multiple splits:")
split_counts = manifest.groupby("image")["split"].nunique()
print((split_counts > 1).sum())

Total rows: 12064
Unique image filenames: 12064
Duplicate filenames: 0

Images per split:
split
train         9651
test          1207
validation    1206
Name: count, dtype: int64

Images assigned to multiple splits:
0


In [16]:
import zipfile
import xml.etree.ElementTree as ET
import pandas as pd

from sklearn.model_selection import train_test_split
from pathlib import PurePosixPath

archive_path = "/content/drive/MyDrive/DistributedAIProject/MBDD2025.zip"
output_path = "/content/drive/MyDrive/DistributedAIProject/mbdd_classification_manifest.csv"

random_seed = 42
class_names = {"abscission", "bulge", "corrosion", "crack", "leakage"}

records = []
excludedEmpty = 0
excludedMulti = 0
excludedUnknown = 0

with zipfile.ZipFile(archive_path, "r") as f:
    xml_files = [i for i in f.namelist() if i.startswith("MBDD2025/Annotations/") and i.endswith(".xml")]
    for j in xml_files:
        root = ET.fromstring(f.read(j))
        labels = {
            k.findtext("name", "").strip().lower()
            for k in root.findall("object")
            if k.findtext("name", "").strip()
        }

        if not labels:
            excludedEmpty = excludedEmpty+1
            continue

        if not labels.issubset(class_names):
            excludedUnknown = excludedUnknown+1
            continue

        if len(labels) != 1:
            excludedMulti = excludedMulti+1
            continue

        records.append({
            "image": PurePosixPath(j).stem + ".jpg",
            "label": next(iter(labels)),
        })

load = pd.DataFrame(records)

train, temp = train_test_split(load, test_size=0.20, random_state=random_seed, stratify=load["label"])
validation, test = train_test_split(temp, test_size=0.50, random_state=random_seed, stratify=temp["label"])

train = train.copy()
validation = validation.copy()
test = test.copy()

train["split"] = "train"
validation["split"] = "validation"
test["split"] = "test"

manifest = pd.concat([train, validation, test], ignore_index=True)
manifest.to_csv(output_path, index=False)

print("Annotations:", len(xml_files))
print("Excluded empty:", excludedEmpty)
print("Excluded multi-category:", excludedMulti)
print("Excluded unknown labels:", excludedUnknown)
print("\nSplit sizes:")
print(manifest["split"].value_counts())
print("\nClass distribution:")
print(pd.crosstab(manifest["label"], manifest["split"]))
print("\nSaved permanently to:", output_path)

Annotations: 14471
Excluded empty: 8
Excluded multi-category: 2399
Excluded unknown labels: 0

Split sizes:
split
train         9651
test          1207
validation    1206
Name: count, dtype: int64

Class distribution:
split       test  train  validation
label                              
abscission   334   2672         334
bulge         75    600          75
corrosion    166   1330         166
crack        437   3490         436
leakage      195   1559         195

Saved permanently to: /content/drive/MyDrive/DistributedAIProject/mbdd_classification_manifest.csv


In [17]:
# Duplicate Check
import pandas as pd
import zipfile
import hashlib

archive_path = ("/content/drive/MyDrive/DistributedAIProject/MBDD2025.zip")
manifest_path = (
    "/content/drive/MyDrive/DistributedAIProject/"
    "mbdd_classification_manifest.csv"
)

manifest = pd.read_csv(manifest_path)
hashRecords = []

with zipfile.ZipFile(archive_path, "r") as f:
  for i, j in enumerate(manifest.itertuples(index=False), start=1):
    imagePath = "MBDD2025/JPEGImages/"+j.image
    with f.open(imagePath) as imageFile:
      digest = hashlib.sha256(imageFile.read()).hexdigest()
    hashRecords.append(
        {
            "image": j.image,
            "split": j.split,
            "label": j.label,
            "sha256": digest,
        }
    )

    if i%2000 == 0:
      print(f"Checked {i}/{len(manifest)} images")

hash_load = pd.DataFrame(hashRecords)

Checked 2000/12064 images
Checked 4000/12064 images
Checked 6000/12064 images
Checked 8000/12064 images
Checked 10000/12064 images
Checked 12000/12064 images


In [20]:
import zipfile
import hashlib
import pandas as pd

from pathlib import Path

PROJECT_DIR = Path(
    "/content/drive/MyDrive/DistributedAIProject"
)

archive_path = PROJECT_DIR / "MBDD2025.zip"
manifest_path = PROJECT_DIR / "mbdd_classification_manifest.csv"
checkpoint_path = PROJECT_DIR / "image_hash_checkpoint.csv"

manifest = pd.read_csv(manifest_path)

# Resume from the existing checkpoint, if available
if checkpoint_path.exists():
    saved = pd.read_csv(checkpoint_path)
    hashRecords = saved.to_dict("records")
    completed = {i["image"] for i in hash_records}
    print("Resuming from checkpoint:", len(completed), "images")

else:
    hashRecords = []
    completed = set()
    print("Starting a new hash check")

with zipfile.ZipFile(archive_path, "r") as f:
    for i, j in enumerate(manifest.itertuples(index=False), start=1):
        if j.image in completed:
            continue

        image_path = "MBDD2025/JPEGImages/" + j.image

        with f.open(image_path) as image_file:
            digest = hashlib.sha256(image_file.read()).hexdigest()

        hashRecords.append(
            {
                "image": j.image,
                "split": j.split,
                "label": j.label,
                "sha256": digest,
            }
        )

        completed.add(j.image)

        if len(hashRecords)%500 == 0:
            pd.DataFrame(hashRecords).to_csv(checkpoint_path, index=False)
            print("Checkpoint saved:", len(hashRecords))

# Save the final complete result
hash_load = pd.DataFrame(hashRecords)
hash_load.to_csv(checkpoint_path, index=False)

print("\nImages hashed:", len(hash_load))
print("Checkpoint saved to:", checkpoint_path)

Starting a new hash check
Checkpoint saved: 500
Checkpoint saved: 1000
Checkpoint saved: 1500
Checkpoint saved: 2000
Checkpoint saved: 2500
Checkpoint saved: 3000
Checkpoint saved: 3500
Checkpoint saved: 4000
Checkpoint saved: 4500
Checkpoint saved: 5000
Checkpoint saved: 5500
Checkpoint saved: 6000
Checkpoint saved: 6500
Checkpoint saved: 7000
Checkpoint saved: 7500
Checkpoint saved: 8000
Checkpoint saved: 8500
Checkpoint saved: 9000
Checkpoint saved: 9500
Checkpoint saved: 10000
Checkpoint saved: 10500
Checkpoint saved: 11000
Checkpoint saved: 11500
Checkpoint saved: 12000

Images hashed: 12064
Checkpoint saved to: /content/drive/MyDrive/DistributedAIProject/image_hash_checkpoint.csv


In [22]:
# Checking cross-split duplicates
import pandas as pd

hash_load = pd.read_csv(
    "/content/drive/MyDrive/DistributedAIProject/"
    "image_hash_checkpoint.csv"
)

duplicate_groups = hash_load.groupby("sha256").filter(lambda group: group["split"].nunique() > 1)

print("Images hashed:", len(hash_load))
print("Images in cross-split exact-duplicate groups:", duplicate_groups["image"].nunique())

if not duplicate_groups.empty:
    display(duplicate_groups.sort_values("sha256"))

hash_load.to_csv("/content/drive/MyDrive/DistributedAIProject/" "image_hashes.csv", index=False,)
print("Final hash results saved.")

Images hashed: 12064
Images in cross-split exact-duplicate groups: 109


,image,split,label,sha256
11122,Hefei4365.jpg,test,bulge,00886662c03a7fa67f772f27401b309eaf75f3dc6ee3ea...
5935,Hefei4009.jpg,train,bulge,00886662c03a7fa67f772f27401b309eaf75f3dc6ee3ea...
6665,Hefei4032.jpg,train,bulge,0a5299c71ebeb7d6f552ceed3e848690c09638892d109f...
10935,Hefei4434.jpg,test,bulge,0a5299c71ebeb7d6f552ceed3e848690c09638892d109f...
1755,Hefei3938.jpg,train,bulge,0df48f46f3f7f58ad8f3360dbe2f4656ab554f4383c842...
...,...,...,...,...
10631,Hefei3969.jpg,validation,bulge,f78776572a42cc7b8731ae55725eb6d31af0ddec67d757...
348,Hefei4017.jpg,train,bulge,fddc670e6db308f505ce3b17e333635d997525c8c494e9...
11134,Hefei4389.jpg,test,bulge,fddc670e6db308f505ce3b17e333635d997525c8c494e9...
9864,Hefei4185.jpg,validation,bulge,feff4608ea2d91ce2e48440749ba23f4053c53fb6d21e2...


Final hash results saved.


In [24]:
# Building duplicate groups
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/DistributedAIProject"

hash_load = pd.read_csv(f"{PROJECT_DIR}/image_hashes.csv")
manifest = pd.read_csv(f"{PROJECT_DIR}/mbdd_classification_manifest.csv")

# Each identical file-content group receives one group ID
hash_load["duplicate_group"] = (hash_load.groupby("sha256", sort=False).ngroup())

# Counting group that span more than one split
group_summary = hash_load.groupby("duplicate_group").agg(
    image_count=("image", "size"),
    split_count=("split", "nunique"),
    splits=("split", lambda x: ", ".join(sorted(set(x)))),
    labels=("label", lambda x: ", ".join(sorted(set(x)))),
)

cross_split_groups = group_summary[group_summary["split_count"] > 1].sort_values("image_count", ascending=False)

print("Total images: ", len(hash_load))
print("Unique content groups: ", hash_load["sha256"].nunique())
print("Groups spanning multiple splits: ", len(cross_split_groups))
print("Images in those groups: ", int(cross_split_groups["image_count"].sum()))

display(cross_split_groups.head(20))
cross_split_groups.to_csv(f"{PROJECT_DIR}/cross_split_duplicate_groups.csv")

print("\nDuplicate group report saved to Google Drive.")

Total images:  12064
Unique content groups:  11896
Groups spanning multiple splits:  54
Images in those groups:  109


,image_count,split_count,splits,labels
duplicate_group,,,,
5112,3,2,"test, train",bulge
62,2,2,"train, validation",bulge
312,2,2,"test, train",bulge
348,2,2,"test, train",bulge
1171,2,2,"train, validation",bulge
175,2,2,"test, train",bulge
1590,2,2,"test, train",bulge
1751,2,2,"test, train",bulge
1954,2,2,"train, validation",bulge



Duplicate group report saved to Google Drive.


In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_DIR = "/content/drive/MyDrive/DistributedAIProject"

manifest = pd.read_csv(f"{PROJECT_DIR}/mbdd_classification_manifest.csv")
hash_load = pd.read_csv(f"{PROJECT_DIR}/image_hashes.csv")

# Attach each image's content hash to the manifest
data = manifest.merge(
    hash_load[["image", "sha256"]],
    on="image",
    how="left",
    validate="one_to_one",
)

assert data["sha256"].notna().all()

# Every identical-content group must remain together
groups = (data.groupby("sha256").agg(label=("label", lambda s: s.mode().iloc[0]), size=("image", "size"),).reset_index())

# Stratify groups by class, weighted approximately by image count.
# Group-level stratification is approximate because duplicate groups can contain different numbers of images.
train_groups, temp_groups = train_test_split(groups, test_size=0.20, random_state=42, stratify=groups["label"],)
validation_groups, test_groups = train_test_split(temp_groups, test_size=0.50, random_state=42, stratify=temp_groups["label"],)

split_by_hash = {}
for i, j in [(train_groups, "train"), (validation_groups, "validation"), (test_groups, "test"),]:
    split_by_hash.update(dict(zip(i["sha256"], [j]*len(i))))

data["split"] = data["sha256"].map(split_by_hash)

# Verifying that exact duplicates do not cross splits
assert data.groupby("sha256")["split"].nunique().max() == 1

corrected = data[["image", "label", "split"]].copy()
output_path = f"{PROJECT_DIR}/mbdd_classification_manifest_deduplicated.csv"
corrected.to_csv(output_path, index=False)

print("Images:", len(corrected))
print("\nSplit sizes:")
print(corrected["split"].value_counts())
print("\nClass distribution:")
print(pd.crosstab(corrected["label"], corrected["split"]))
print("\nCross-split exact-duplicate groups:",
      int((data.groupby("sha256")["split"].nunique() > 1).sum()))
print("\nSaved:", output_path)

Images: 12064

Split sizes:
split
train         9647
validation    1214
test          1203
Name: count, dtype: int64

Class distribution:
split       test  train  validation
label                              
abscission   334   2672         334
bulge         71    597          82
corrosion    166   1329         167
crack        437   3490         436
leakage      195   1559         195

Cross-split exact-duplicate groups: 0

Saved: /content/drive/MyDrive/DistributedAIProject/mbdd_classification_manifest_deduplicated.csv


In [27]:
# Verifying that the manifest has been saved in Drive
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/DistributedAIProject"

manifest = pd.read_csv(f"{PROJECT_DIR}/mbdd_classification_manifest_deduplicated.csv")
hash_load = pd.read_csv(f"{PROJECT_DIR}/image_hashes.csv")

check = manifest.merge(hash_load[["image", "sha256"]], on="image", validate="one_to_one",)
cross_split_groups = (check.groupby("sha256")["split"].nunique() > 1).sum()

print("Manifest rows: ", len(manifest))
print("Unique filenames: ", manifest["image"].nunique())
print("Missing labels: ", manifest["label"].isna().sum())
print("Cross-split exact-duplicate groups: ", cross_split_groups)

assert len(manifest) == 12064
assert manifest["image"].nunique() == 12064
assert cross_split_groups == 0

print("\nValidation passed!")

Manifest rows:  12064
Unique filenames:  12064
Missing labels:  0
Cross-split exact-duplicate groups:  0

Validation passed!


In [28]:
# Inspecting near duplicate checks
import pandas as pd
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/DistributedAIProject")
manifest = pd.read_csv(PROJECT_DIR / "mbdd_classification_manifest_deduplicated.csv")

In [29]:
# Extracting numeric suffixes from filenames
manifest["sequence_number"] = (manifest["image"].str.extract(r"Hefei(\d+)", expand=False).astype("Int64"))
manifest = manifest.sort_values("sequence_number")

print("Images: ", len(manifest))
print("Images with parsed Hefei numbers: ", manifest["sequence_number"].notna().sum())

Images:  12064
Images with parsed Hefei numbers:  12064


In [30]:
# Inspecting nearby filenames whose images are assigned to different splits
manifest["previous_number"] = manifest["sequence_number"].shift()
manifest["previous_split"] = manifest["split"].shift()
manifest["number_gap"] = (manifest["sequence_number"]-manifest["previous_number"])

nearby_cross_split = manifest[(manifest["number_gap"] > 0) & (manifest["number_gap"] <= 5) & (manifest["split"] != manifest["previous_split"])]

print("\nNearby filename numbers crossing split boundaries:")
display(nearby_cross_split[["image", "sequence_number", "split", "previous_number", "previous_split", "number_gap"]].head(30))


Nearby filename numbers crossing split boundaries:


,image,sequence_number,split,previous_number,previous_split,number_gap
8432,Hefei2.jpg,2,validation,1,train,1
5859,Hefei3.jpg,3,train,2,validation,1
8152,Hefei10.jpg,10,test,9,train,1
11203,Hefei11.jpg,11,train,10,test,1
6346,Hefei15.jpg,15,validation,14,train,1
11182,Hefei16.jpg,16,train,15,validation,1
5742,Hefei21.jpg,21,validation,20,train,1
1856,Hefei23.jpg,23,train,22,validation,1
11946,Hefei43.jpg,43,test,42,train,1
2702,Hefei44.jpg,44,train,43,test,1


In [31]:
# Searching for images automatically
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image

In [36]:
import pandas as pd

project_dir = Path("/content/drive/MyDrive/DistributedAIProject")
manifest_path = project_dir / "mbdd_classification_manifest_deduplicated.csv"

load = pd.read_csv(manifest_path)

print("Shape:", load.shape)
print("\nColumns:")
print(load.columns.tolist())

print("\nFirst 10 rows:")
display(load.head(10))

print("\nFilename examples:")
filename = next((i for i in load.columns if "filename" in i.lower() or "image" in i.lower()), None)
if filename:
    print(load[filename].astype(str).sample(min(20, len(load)), random_state=42).to_list())

else:
    print("No obvious filename column found.")

print("\nClass counts:")
for i in load.columns:
    if any(j in i.lower() for j in ["label", "class", "category", "defect"]):
        print(f"\n{i}:")
        print(load[i].value_counts(dropna=False).to_string())

print("\nSplit counts:")
for i in load.columns:
    if "split" in i.lower():
        print(load[i].value_counts(dropna=False).to_string())


Shape: (12064, 3)

Columns:
['image', 'label', 'split']

First 10 rows:


,image,label,split
0,Hefei1750.jpg,crack,train
1,Hefei3568.jpg,crack,test
2,Hefei206.jpg,corrosion,train
3,Hefei7694.jpg,crack,train
4,Hefei2910.jpg,leakage,train
5,Hefei7075.jpg,leakage,test
6,Hefei5711.jpg,crack,test
7,Hefei5512.jpg,crack,train
8,Hefei3628.jpg,crack,train
9,Hefei8755.jpg,abscission,train



Filename examples:
['Hefei7104.jpg', 'Hefei4957.jpg', 'Hefei11994.jpg', 'Hefei327.jpg', 'Hefei2465.jpg', 'Hefei183.jpg', 'Hefei5643.jpg', 'Hefei5613.jpg', 'Hefei9450.jpg', 'Hefei4969.jpg', 'Hefei4173.jpg', 'Hefei10515.jpg', 'Hefei1156.jpg', 'Hefei9689.jpg', 'Hefei3063.jpg', 'Hefei88.jpg', 'Hefei5083.jpg', 'Hefei11719.jpg', 'Hefei3404.jpg', 'Hefei4805.jpg']

Class counts:

label:
label
crack         4363
abscission    3340
leakage       1949
corrosion     1662
bulge          750

Split counts:
split
train         9647
validation    1214
test          1203


In [37]:
import pandas as pd
import numpy as np
import re

from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold

project_dir = Path("/content/drive/MyDrive/DistributedAIProject")
input_path = project_dir / "mbdd_classification_manifest_deduplicated.csv"
output_path = project_dir / "mbdd_sequence_split_candidate.csv"

load = pd.read_csv(input_path).copy()

In [38]:
# Extract the numeric sequence ID from filenames such as Hefei1750.jpg
def get_sequence_number(filename):
    match = re.fullmatch(r"Hefei(\d+)\.jpg", str(filename), flags=re.IGNORECASE)
    return int(match.group(1)) if match else None

load["sequence_number"] = load["image"].apply(get_sequence_number)

if load["sequence_number"].isna().any():
    raise ValueError("Some filenames do not match the expected Hefei<number>.jpg pattern.")

In [39]:
# Group consecutive numbers together.
# A gap greater than 1 starts a new group.
ordered = load[["image", "sequence_number"]].sort_values("sequence_number")
group_by_image = {}
group_id = -1
previous_number = None

for i in ordered.itertuples(index=False):
    number = int(i.sequence_number)
    if previous_number is None or number != previous_number + 1:
        group_id = group_id+1

    group_by_image[i.image] = group_id
    previous_number = number

load["sequence_group"] = load["image"].map(group_by_image)

In [40]:
# Approximate the original 80/10/10 split using grouped stratification.
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train, temp = next(outer.split(load, y=load["label"], groups=load["sequence_group"]))

load["new_split"] = ""
load.loc[train, "new_split"] = "train"

temp = load.iloc[temp].copy()

In [41]:
# Splitting the temporary 20% approximately equally into validation and test.
inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=43)
validation, test = next(inner.split(temp, y=temp["label"], groups=temp["sequence_group"]))

load.loc[temp.iloc[validation].index, "new_split"] = "validation"
load.loc[temp.iloc[test].index, "new_split"] = "test"

In [42]:
# Save a clean candidate manifest, retaining only the required columns.
candidate = load[["image", "label", "new_split"]].rename(columns={"new_split": "split"})
candidate.to_csv(output_path, index=False)

print("Saved candidate: ", output_path)
print("\nSplit sizes: ")
print(candidate["split"].value_counts())

print("\nClass distribution by split: ")
print(pd.crosstab(candidate["label"], candidate["split"]))

# Verify that no sequence group crosses split boundaries.
check = load.groupby("sequence_group")["new_split"].nunique()
print("\nGroups crossing splits: ", int((check > 1).sum()))
print("Total images: ", len(candidate))
print("Unique filenames: ", candidate["image"].nunique())

Saved candidate:  /content/drive/MyDrive/DistributedAIProject/mbdd_sequence_split_candidate.csv

Split sizes: 
split
train         9222
validation    2774
test            68
Name: count, dtype: int64

Class distribution by split: 
split       test  train  validation
label                              
abscission    68   2523         749
bulge          0      0         750
corrosion      0   1662           0
crack          0   3684         679
leakage        0   1353         596

Groups crossing splits:  0
Total images:  12064
Unique filenames:  12064


In [45]:
import pandas as pd
import re
from pathlib import Path

project_dir = Path("/content/drive/MyDrive/DistributedAIProject")
load = pd.read_csv(project_dir / "mbdd_classification_manifest_deduplicated.csv")

load["sequence_number"] = load["image"].str.extract(r"^Hefei(\d+)\.jpg$", expand=False).astype(int)

ordered = load.sort_values("sequence_number").copy()
ordered["gap"] = ordered["sequence_number"].diff()
ordered["group_id"] = ordered["gap"].ne(1).cumsum()

groups = ordered.groupby("group_id").agg(
    group_size=("image", "size"),
    first=("sequence_number", "min"),
    last=("sequence_number", "max"),
    classes=("label", "nunique")
)

print("Number of groups:", len(groups))
print("\nGroup-size summary:")
print(groups["group_size"].describe())

print("\nLargest groups:")
display(groups.sort_values("group_size", ascending=False).head(15))

print("\nClasses in largest groups:")
largest_idx = groups.nlargest(10, "group_size").index
display(ordered[ordered["group_id"].isin(largest_idx)].groupby("group_id").agg(
        size=("image", "size"),
        classes=("label", lambda x: ", ".join(sorted(x.unique()))),
        first=("image", "first"),
        last=("image", "last")
    ).sort_values("size", ascending=False))

Number of groups: 80

Group-size summary:
count      80.000000
mean      150.800000
std       307.079605
min         1.000000
25%        11.750000
50%        34.000000
75%       125.250000
max      1889.000000
Name: group_size, dtype: float64

Largest groups:


,group_size,first,last,classes
group_id,,,,
10,1889,3352,5240,4
12,1509,5397,6905,1
6,838,1822,2659,2
4,707,1018,1724,4
20,677,9029,9705,2
13,632,6936,7567,2
2,475,426,900,2
1,424,1,424,1
19,361,8667,9027,2



Classes in largest groups:


,size,classes,first,last
group_id,,,,
10,1889,"abscission, bulge, crack, leakage",Hefei3352.jpg,Hefei5240.jpg
12,1509,crack,Hefei5397.jpg,Hefei6905.jpg
6,838,"crack, leakage",Hefei1822.jpg,Hefei2659.jpg
4,707,"abscission, corrosion, crack, leakage",Hefei1018.jpg,Hefei1724.jpg
20,677,"abscission, crack",Hefei9029.jpg,Hefei9705.jpg
13,632,"abscission, leakage",Hefei6936.jpg,Hefei7567.jpg
2,475,"corrosion, leakage",Hefei426.jpg,Hefei900.jpg
1,424,corrosion,Hefei1.jpg,Hefei424.jpg
19,361,"abscission, leakage",Hefei8667.jpg,Hefei9027.jpg


## Final Validation Summary

### Dataset
- Dataset: MBDD2025
- Total images selected: 12,064
- Task: 5-class, single-label image classification
- Classes: abscission, bulge, corrosion, crack, leakage
- Annotation source: VOC XML annotations

### Dataset preparation and validation
- Selected images with exactly one distinct damage category.
- Excluded multi-category images from this single-label experiment.
- Generated an 80/10/10 train-validation-test split.
- Computed SHA-256 hashes to detect exact duplicate images.
- Corrected the split to prevent exact duplicate images from crossing splits.
- Verified 12,064 rows, unique filenames, no missing values, and all five classes across the splits.

### Final split
- Train: 9,647 images
- Validation: 1,214 images
- Test: 1,203 images

### Limitations
- Consecutive Hefei filenames showed potentially related captures in different splits.
- Filename adjacency alone does not establish visual similarity.
- A naive sequence-grouping attempt produced a severely imbalanced split and was rejected.
- Exact duplicate leakage was checked; all possible near-duplicate leakage has not been ruled out.

### Next stage
Prepare reproducible image preprocessing and train the custom CNN baseline, followed by ResNet18 using the same split and evaluation protocol.